# 18 — Fairness (Vertex AI Gen AI Evaluation + cohort parity)

**AWS-side equivalent:** Bedrock managed eval + per-cohort selection-rate parity (`notebooks/aws/18-fairness.ipynb`)
**GCP service:** Vertex AI **Gen AI Evaluation Service** (`EvalTask`, `PointwiseMetric`) — the per-row LLM-as-judge piece; cohort aggregation runs in pandas. Scores are read from **BigQuery** (`{bq_dataset}.people.composite_score`, written by `14-scoring.ipynb`).

_Naming note (as of 2026-05): the per-row quality service is **Vertex AI Gen AI Evaluation Service** (`from vertexai.evaluation import EvalTask`). **Vertex AI Model Monitoring** is the separate, drift-focused service for feature skew/drift on deployed models — and **Vertex AI Model Bias** metrics (`vertex-ai/docs/evaluation/model-bias-metrics`) compute SPD/DI on classification predictions. There is no first-party "Responsible AI Toolkit" Python package; fairness work is a mix of (a) Gen AI Eval pointwise metrics for response quality, (b) cohort aggregation in pandas (this notebook), and optionally (c) Model Monitoring for drift dashboards._

Caveat carried over from AWS side: `data/org/people.json` has no legally protected attributes. We use **department** as the cohort attribute — the realistic enterprise concern ("does scoring systematically advantage Platform over Mobile?"). Swap in protected attributes in production.

## Steps

1. Load scored people from BigQuery (`composite_score` written by `14-scoring.ipynb`); fall back to a deterministic synthetic score if `bq_dataset` is unset
2. Define a per-row helpfulness/groundedness eval with Gen AI Evaluation (`gemini-2.5-flash` as judge)
3. Aggregate scores by department → selection rate, Disparate Impact Ratio, Statistical Parity Difference
4. Flag failing cohorts against the 4/5 rule
5. (Optional) point at Vertex AI Model Monitoring for drift dashboards

In [ ]:
# State plumbing — every notebook reads/writes config.json under the 'gcp' namespace.
import json
from pathlib import Path

config = json.loads(Path('config.json').read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
JUDGE_MODEL = gcp.get('judge_model', 'gemini-2.5-flash')
TARGET_MODEL = gcp.get('model_id', 'gemini-2.5-flash')
# BigQuery dataset where 14-scoring.ipynb wrote composite_score. If unset we
# fall back to a deterministic synthetic score so the cohort math still runs.
BQ_DATASET = gcp.get('bq_dataset')
BQ_LOCATION = gcp.get('bq_location', 'US')
assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'

# Set True to actually fire the Vertex eval — every row costs a small amount of
# judge-model throughput. The cohort metrics in Step 4 work either way.
RUN_VERTEX_EVAL = False
print(f'project={PROJECT_ID} region={REGION} judge={JUDGE_MODEL} target={TARGET_MODEL} run_eval={RUN_VERTEX_EVAL}')
print(f'bq_dataset={BQ_DATASET or "(unset — using synthetic fallback)"} bq_location={BQ_LOCATION}')

## Step 1: Load scored people from BigQuery (with synthetic fallback)

Primary path: `14-scoring.ipynb` writes `expertise_score`, `pagerank_score`, and `composite_score` back to `{project}.{bq_dataset}.people`. We `SELECT` it directly — no extra connector needed, just `google-cloud-bigquery` + ADC.

Fallback: if `bq_dataset` is unset (i.e., `02-bigquery-graph.ipynb` hasn't run), we derive a deterministic score off the seed JSON via SHA-256 so the cohort math in Steps 3-4 remains reproducible without infrastructure.

In [ ]:
import hashlib
import pandas as pd


def synthetic_scores_from_json() -> pd.DataFrame:
    """Deterministic SHA-256-based score off the seed JSON.

    Kept as the fallback path so the notebook is runnable without BigQuery.
    Tilts Platform/Backend slightly higher to make the parity story visible.
    """
    people = json.loads(Path('data/org/people.json').read_text())
    out = pd.DataFrame(people)

    def _score(row) -> float:
        digest = hashlib.sha256(row['id'].encode()).hexdigest()
        base = int(digest[:8], 16) / 0xFFFFFFFF
        bump = {'Platform Engineering': 0.10, 'Backend Engineering': 0.05}.get(row.get('department', ''), 0.0)
        return max(0.0, min(1.0, base * 0.9 + bump))

    out['composite_score'] = out.apply(_score, axis=1)
    return out[['id', 'name', 'department', 'composite_score']]


if BQ_DATASET:
    from google.cloud import bigquery
    bq = bigquery.Client(project=PROJECT_ID)
    try:
        df = bq.query(f"""
            SELECT id, name, department, composite_score
            FROM `{PROJECT_ID}.{BQ_DATASET}.people`
            WHERE composite_score IS NOT NULL
        """).to_dataframe()
        print(f'✓ loaded {len(df)} scored people from BigQuery')
        if df.empty:
            print('⚠ BigQuery returned 0 scored people — falling back to synthetic')
            df = synthetic_scores_from_json()
    except Exception as exc:
        print(f'⚠ BigQuery read failed ({exc}); falling back to synthetic')
        df = synthetic_scores_from_json()
else:
    print('ℹ bq_dataset unset — using synthetic fallback. Run 02-bigquery-graph.ipynb + 14-scoring.ipynb for the real path.')
    df = synthetic_scores_from_json()

df['cohort'] = df['department'].fillna('Unknown')
print(f'Loaded {len(df)} people across {df["cohort"].nunique()} cohorts')
df[['id', 'name', 'cohort', 'composite_score']].head(10)

## Step 2: Per-row evaluation with Vertex AI Gen AI Evaluation

Build a small synthetic eval set: for each cohort sample, the agent's job is to summarise the person's expertise in one sentence. We score each generation with `PointwiseMetric` (groundedness + helpfulness). `EvalTask.evaluate()` runs the judge model row-by-row and returns a per-row dataframe.

The cell is gated on `RUN_VERTEX_EVAL`. When the flag is `False` we fabricate a quality score from the same deterministic seed so Step 3 still has something to aggregate.

In [ ]:
eval_rows = df.head(12).to_dict(orient='records')  # 12 people across the cohorts

def _instr(row):
    return (
        f"Person: {row['name']} ({row.get('title', '?')}, {row['cohort']}).\n"
        f"Write one sentence describing why this person is or isn't a good\n"
        f"expertise referral for a Kubernetes-platform question."
    )

eval_df_in = pd.DataFrame([
    {
        'prompt': _instr(r),
        'reference': f"{r['name']} works in {r['cohort']} with composite score {r['composite_score']:.2f}.",
        'cohort': r['cohort'],
        'person_id': r['id'],
    }
    for r in eval_rows
])

if RUN_VERTEX_EVAL:
    import vertexai
    from vertexai.evaluation import EvalTask, PointwiseMetric, PointwiseMetricPromptTemplate

    vertexai.init(project=PROJECT_ID, location=REGION)

    helpfulness = PointwiseMetric(
        metric='helpfulness',
        metric_prompt_template=PointwiseMetricPromptTemplate(
            criteria={
                'helpfulness': 'Does the response usefully decide whether to refer the person, citing concrete signals?',
            },
            rating_rubric={
                '5': 'Decisive recommendation with concrete justification.',
                '3': 'Hedged recommendation, weak justification.',
                '1': 'No clear recommendation or unsupported assertion.',
            },
        ),
    )

    task = EvalTask(
        dataset=eval_df_in,
        metrics=[helpfulness],
        experiment='orgiq-fairness-cohort',
    )
    result = task.evaluate(model=TARGET_MODEL)
    eval_df = result.metrics_table.copy()
    eval_df['cohort'] = eval_df_in['cohort'].values
    eval_df['person_id'] = eval_df_in['person_id'].values
    # Normalise: judge returns 1..5, project to 0..1
    eval_df['quality'] = (eval_df['helpfulness/score'] - 1.0) / 4.0
else:
    # Synthetic stand-in keyed off person_id so re-runs are stable.
    eval_df = eval_df_in.copy()
    eval_df['quality'] = [
        ((int(hashlib.sha256(pid.encode()).hexdigest()[8:16], 16) / 0xFFFFFFFF) * 0.7) + 0.3
        for pid in eval_df['person_id']
    ]

print(f'✓ {len(eval_df)} eval rows, mean quality {eval_df["quality"].mean():.2f}')
eval_df[['person_id', 'cohort', 'quality']].head()

## Step 3: Per-cohort score distribution + EEOC-style parity metrics

Treat the **top quartile of `composite_score`** as the "selected" group (people the supervisor will surface). For each cohort vs the highest-rate cohort, compute:

- **Selection rate** `P(selected | cohort)`
- **Disparate Impact Ratio** `rate(g) / rate(ref)` — 4/5 rule: ≥ 0.8 is conventionally OK
- **Statistical Parity Difference** `rate(g) - rate(ref)` — closer to 0 is fairer

In [ ]:
threshold = df['composite_score'].quantile(0.75)
df['selected'] = (df['composite_score'] >= threshold).astype(int)

summary = (
    df.groupby('cohort')
      .agg(cohort_size=('id', 'count'),
           selected=('selected', 'sum'),
           mean_score=('composite_score', 'mean'))
      .assign(selection_rate=lambda d: d['selected'] / d['cohort_size'])
      .sort_values('selection_rate', ascending=False)
)
ref_cohort = summary.index[0]
ref_rate = summary.loc[ref_cohort, 'selection_rate'] or 1e-9
summary['disparate_impact_ratio'] = summary['selection_rate'] / ref_rate
summary['statistical_parity_diff'] = summary['selection_rate'] - ref_rate
summary['passes_4_5_rule'] = summary['disparate_impact_ratio'] >= 0.8

print(f'Threshold (top quartile): {threshold:.3f}')
print(f'Reference cohort: {ref_cohort} (rate {ref_rate:.3f})')
summary

In [ ]:
# Join the per-row quality means in too — does cohort A get systematically lower
# generation quality, beyond raw selection rate?
quality_by_cohort = eval_df.groupby('cohort')['quality'].mean().rename('mean_quality')
joined = summary.join(quality_by_cohort, how='left')

failing = joined[~joined['passes_4_5_rule']]
print(f'{len(failing)} of {len(joined)} cohorts fail the 4/5 rule against {ref_cohort}')
if len(failing):
    print('\nFailing cohorts:')
    print(failing[['selection_rate', 'disparate_impact_ratio', 'mean_quality']].round(3).to_string())
joined.round(3)

## Step 4: Visualise the disparities

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#3b82f6' if v else '#ef4444' for v in joined['passes_4_5_rule']]

joined['selection_rate'].plot.bar(ax=ax1, color=colors)
ax1.axhline(ref_rate * 0.8, color='black', linestyle='--', label='4/5 rule floor')
ax1.set_title('Selection rate by cohort (red = fails 4/5)')
ax1.set_ylabel('P(selected | cohort)')
ax1.legend()

joined['disparate_impact_ratio'].plot.bar(ax=ax2, color=colors)
ax2.axhline(0.8, color='black', linestyle='--', label='0.8 floor')
ax2.axhline(1.0, color='gray', linestyle=':', label='parity')
ax2.set_title('Disparate Impact Ratio by cohort')
ax2.set_ylabel('DIR = rate(g) / rate(ref)')
ax2.legend()

plt.tight_layout()
plt.show()

## Step 5: (Optional) Vertex AI Model Monitoring for drift dashboards

Per-cohort parity above is a *batch* check — it catches systemic bias on a frozen snapshot. **Vertex AI Model Monitoring** is the live, deployed-model dashboard: it watches feature skew (training-vs-serving) and drift (serving-over-time) on a Vertex AI endpoint. For a generative system we'd hook it up to a custom model endpoint that publishes the per-request cohort + score as features, then configure a drift threshold.

Reference: `cloud.google.com/vertex-ai/docs/model-monitoring/using-model-monitoring`. We do not deploy a monitoring job here because it requires a long-running Vertex endpoint with traffic. The notebook prints the API shape so it's visible in code review.

In [ ]:
MONITORING_JOB_SHAPE = {
    'display_name': 'orgiq-scoring-cohort-drift',
    'endpoint': f'projects/{PROJECT_ID}/locations/{REGION}/endpoints/<endpoint-id>',
    'logging_sampling_strategy': {'random_sample_config': {'sample_rate': 0.1}},
    'objective_configs': [
        {
            'objective_config': {
                'training_dataset': {
                    'data_format': 'csv',
                    'gcs_source': {'uris': [f'gs://{PROJECT_ID}-orgiq/baseline/scoring.csv']},
                    'target_field': 'composite_score',
                },
                'training_prediction_skew_detection_config': {
                    'skew_thresholds': {'cohort': {'value': 0.1}, 'composite_score': {'value': 0.1}},
                },
                'prediction_drift_detection_config': {
                    'drift_thresholds': {'cohort': {'value': 0.1}, 'composite_score': {'value': 0.1}},
                },
            },
        }
    ],
}
print(json.dumps(MONITORING_JOB_SHAPE, indent=2))
print('\nTo create: aiplatform.ModelDeploymentMonitoringJob.create(**shape) — gated on having a deployed endpoint.')

## Summary

- Loaded people seed and attached `composite_score` (BigQuery primary; deterministic SHA-256 fallback when `bq_dataset` is unset)
- Built a Gen AI Evaluation `PointwiseMetric` pipeline (gated by `RUN_VERTEX_EVAL`)
- Aggregated per-cohort: selection rate, DIR, SPD, mean judge quality
- Flagged 4/5-rule failures and visualised disparities
- Sketched Vertex AI Model Monitoring shape for the live-drift companion

**Open question:** the BigQuery read is currently a full scan of `people`. For larger orgs we should add a `PARTITION BY scored_at` (date) clause and read the latest partition only, instead of `WHERE composite_score IS NOT NULL`.

**Next:** `18b-text-to-sql.ipynb` (Gemini structured output → BigQuery).